# Step 4 — KGIN re-ranker

Re-ranks each search's list with KGIN, adapted to this data. Each change targets a weakness found in step 3.

**Setup.** Same as steps 1–3: 70/15/15 split by search (seed 4222), full lists, NDCG@10 with gains 0 / 1 / 3 / 7, and ties broken at random. Validation picks the settings, and test is scored once, in Section 6. Every search is on 17 Oct with no time of day, so other users' data is only used if it's from before that day.

**Where to find the results.**
- Train and validation: end of Section 4
- Ablations and KGIN's gain over the features: end of Section 5
- Test and paired tests: Section 6

| Section | Content | Output (`model/`) |
|---|---|---|
| 0 | Setup | — |
| 1 | Knowledge graph and interaction graph | — |
| 2 | Model inputs | — |
| 3 | Model | — |
| 4 | Training | `kgin.pt`, `features_mlp.pt`, `kgin_features.pt`, two same-day checks, `score_*.parquet` |
| 5 | Ablations | `ablations.csv`, `kgin_gain_check.csv` |
| 6 | Test results | `results.csv`, `paired_tests.csv` |
| 7 | Interpretation | `intent_weights.csv`, `segments.csv` |

## Section 0 · Setup

- Paths, constants and evaluation helpers.
- `ndcg_by_search`: NDCG@10 of each search (same metric as steps 2–3). `ndcg_at_k`: its mean.
- `evaluate`: train and validation NDCG@10. `save_scores` / `load_scores`: one score per shown product, in exposures order.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, json, time, copy, psutil
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
DEV = "cuda"

ROOT  = "/content/drive/MyDrive/BT4222_Group10_JDSearch"
OUT   = f"{ROOT}/processed"     # step-1 tables
FEAT  = f"{ROOT}/features"      # step-2 features
BASE  = f"{ROOT}/baselines"     # step-3 scores
MODEL = f"{ROOT}/model"         # this notebook's output
os.makedirs(MODEL, exist_ok=True)

meta = json.load(open(f"{OUT}/meta.json"))
SEED, DAY = meta["seed"], 86_400
LABEL = meta["label_codes"]                          # none 0, click 1, cart 2, order 3
TYPE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}   # history action codes
SPLITS = {"train": 0, "valid": 1, "test": 2}
GAIN = np.array([0.0, 1.0, 3.0, 7.0])                # gain per label: 2^label - 1
SEP = "\x18"                                         # separator between word ids in queries and titles
N_USERS = meta["table_rows"]["searches"]             # one user per search

def ram(tag=""):
    print(f"[{tag}] RAM in use: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

def ndcg_by_search(sid, label, score, k=10, seed=0):
    """NDCG@k of each search. Rank by score (high first, NaN last, ties at random); searches with no interaction are dropped."""
    d = pd.DataFrame({"sid": sid, "label": label, "score": score})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["sid", "score", "tie"], ascending=[True, False, True], na_position="last")
    pos = d.groupby("sid").cumcount().to_numpy()                   # 0 = top
    dcg = pd.Series(np.where(pos < k, GAIN[d.label.to_numpy()] / np.log2(pos + 2), 0.0)).groupby(d.sid.to_numpy()).sum()
    # finds ideal DCG
    best = d.sort_values(["sid", "label"], ascending=[True, False])
    bpos = best.groupby("sid").cumcount().to_numpy()
    idcg = pd.Series(np.where(bpos < k, GAIN[best.label.to_numpy()] / np.log2(bpos + 2), 0.0)).groupby(best.sid.to_numpy()).sum()
    ok = idcg > 0
    return dcg[ok] / idcg[ok] # NDCG

def ndcg_at_k(sid, label, score, k=10, seed=0):
    """Mean NDCG@k over searches (same metric as steps 2-3)."""
    return float(ndcg_by_search(sid, label, score, k, seed).mean())

ex = pd.read_parquet(f"{OUT}/exposures.parquet")      # sid, wid, label, split
SID, LAB = ex.sid.to_numpy(), ex.label.to_numpy()
assert (np.diff(SID) >= 0).all()                      # rows are grouped by search
IN_SPLIT = {name: (ex.split == code).to_numpy() for name, code in SPLITS.items()}
results = []                                          # one row per model

def evaluate(name, score):
    """NDCG@10 on train and validation. Test is scored once, in Section 6."""
    row = {"model": name}
    for split in ["train", "valid"]:
        m = IN_SPLIT[split]
        row[split] = ndcg_at_k(SID[m], LAB[m], score[m]) # use boolean mask indexing
    results[:] = [r for r in results if r["model"] != name] + [row]   # re-running a cell replaces the old row
    print(f"{name:>22s}   train {row['train']:.4f} | valid {row['valid']:.4f}")
    return row

def save_scores(name, score):
    """One score per shown product, in exposures order."""
    ex[["sid", "wid"]].assign(score=np.asarray(score, dtype="float32")).to_parquet(f"{MODEL}/score_{name}.parquet", index=False)

def load_scores(folder, name):
    """Saved scores of a model, checked to line up with exposures."""
    s = pd.read_parquet(f"{folder}/score_{name}.parquet")
    assert (s.sid.to_numpy() == SID).all() and (s.wid.to_numpy() == ex.wid.to_numpy()).all()
    return s.score.to_numpy()

print(f"exposures {len(ex):,} | " + " | ".join(f"{k} searches {ex.sid[m].nunique():,}" for k, m in IN_SPLIT.items()))
ram("setup")

Mounted at /content/drive
exposures 15,510,012 | train searches 120,209 | valid searches 25,759 | test searches 25,760
[setup] RAM in use: 1.3 GB


## Section 1 · Knowledge graph and interaction graph

**Knowledge graph.** Each product links to its brand, shop, category levels 1–4 and title words (7 relations).
- Brand, shop and category values linked to fewer than 2 products share one "rare" vector.
- Words appearing fewer than 2 times are dropped. Titles keep their first 32 words, queries their first 16. Both use one word table.

**Interaction graph.** Degree = distinct users who clicked, carted or ordered the product, from events at least 1 day old (before 17 Oct), stored as log(1 + users). It replaces product ID vectors, which can't be learned for products missing from training.

No labels are used.

| Relation | Entities | Links |
|---|---|---|
| brand | 155,707 | 11,588,589 |
| shop | 232,409 | 11,488,688 |
| cate1 / cate2 / cate3 / cate4 | 73 / 634 / 5,523 / 6,788 | 11,588,589 each |
| title word | 494,627 | 220,305,753 |

- 12,319,978 products, 289,737,386 links.
- 75.2% of shown products appear in only one list (40.5% of shown rows), and 45.3% of test rows show a product in no training list. An ID vector can't be learned for those products.
- 99.5% of shown rows have attributes, including 99.4% of the 68.2% that CF couldn't score in step 3.
- 99.4% of query words are in the word table.

In [ ]:
MIN_PRODUCTS = 2                # a brand / shop / category needs >= 2 products to get its own vector
MIN_TITLES = 2                  # a word needs >= 2 appearances in titles to get its own vector
TITLE_LEN, QUERY_LEN = 32, 16   # words kept per title / query
CHUNK = 1_000_000               # titles processed per chunk, to limit RAM
ATTRS = {"brand": "brand_id", "shop": "shop_id", "cate1": "cate_id_1", "cate2": "cate_id_2", "cate3": "cate_id_3", "cate4": "cate_id_4"}
STREAMS = {"click": [TYPE["CLICK"]], "cart": [TYPE["CART"], TYPE["FLW"]], "order": [TYPE["ORD"]]}

assert not ex.duplicated(["sid", "wid"]).any()   # a product appears at most once in a list

# Products = everything shown or in a history. A product's index is its position in `wids`.
hist = pd.read_parquet(f"{OUT}/history.parquet", columns=["sid", "wid", "type", "age_s"])
wids = np.union1d(ex.wid.to_numpy(), hist.wid.to_numpy())
hist["p"] = np.searchsorted(wids, hist.wid.to_numpy())
ex_p = np.searchsorted(wids, ex.wid.to_numpy())

# 1a. Product -> brand, shop, category 1-4. Row 0 = missing, 1 = rare, 2+ = own vector.
prods = pd.read_parquet(f"{OUT}/products.parquet").set_index("wid").reindex(wids)
ATTR = np.zeros((len(wids), len(ATTRS)), dtype=np.int32)
n_entities = {}
for k, (rel, col) in enumerate(ATTRS.items()):
    v = prods[col].astype("float64")                                     # NaN = not in the catalogue
    n_products = v.map(v.value_counts())                                  # products sharing this value
    own = pd.Index(np.sort(v[n_products >= MIN_PRODUCTS].unique()))       # values that get their own vector
    ATTR[:, k] = np.select([v.isna(), n_products < MIN_PRODUCTS], [0, 1], own.get_indexer(v) + 2)
    n_entities[rel] = len(own) + 2
del prods; gc.collect()

# 1b. Product -> title words. Queries use the same word table.
def first_words(texts, width):
    """Word ids of each text, first `width` only, one row per text. 0 = empty slot."""
    w = pc.list_slice(pc.split_pattern(texts, SEP), 0, width, return_fixed_size_list=True)
    return pc.fill_null(pc.list_flatten(w), "0").cast("int32").to_numpy().reshape(-1, width)

titles = pq.read_table(f"{OUT}/titles.parquet")                               # wid, title
title_ids = np.concatenate([first_words(titles["title"].slice(a, CHUNK), TITLE_LEN) for a in range(0, len(titles), CHUNK)])
count = pd.Series(title_ids[title_ids > 0]).value_counts()
words = pd.Index(np.sort(count.index[count >= MIN_TITLES]))                   # the word table
n_words = len(words) + 1                                                      # + row 0 = no word

def word_rows(ids):
    """Word ids -> rows of the word table (position + 1). Empty slots and rare or unknown words -> 0."""
    return (words.get_indexer(ids.ravel()) + 1).reshape(ids.shape).astype(np.int32)

TITLE = np.zeros((len(wids), TITLE_LEN), dtype=np.int32)                      # products without a title stay 0
title_p = np.searchsorted(wids, titles["wid"].to_numpy())
for a in range(0, len(titles), CHUNK):
    TITLE[title_p[a:a + CHUNK]] = word_rows(title_ids[a:a + CHUNK])

queries = pq.read_table(f"{OUT}/searches.parquet", columns=["sid", "query"]).sort_by("sid")
assert (queries["sid"].to_numpy() == np.arange(N_USERS)).all()
query_ids = first_words(queries["query"], QUERY_LEN)
QUERY = word_rows(query_ids)                                                  # row = sid
del titles, title_ids, title_p, count, queries; gc.collect()

# 1c. Interaction graph: degree = distinct users who acted on the product, per stream.
#     Events >= 1 day old only (before 17 Oct), so no user sees another user's same-day actions.
old = hist.age_s >= DAY
DEG = np.zeros((len(wids), len(STREAMS)), dtype=np.float32)
for k, types in enumerate(STREAMS.values()):
    acted = hist[old & hist.type.isin(types)].drop_duplicates(["p", "sid"])   # one row per (product, user)
    DEG[:, k] = np.log1p(np.bincount(acted.p, minlength=len(wids)))          # log(1 + users)
del old, acted; gc.collect()

# Summary
cold = ~np.isin(ex.wid.to_numpy(), np.load(f"{BASE}/mf_vocab.npy"))         # no column in step 3's user x product grid
has_attr = (ATTR[ex_p] > 0).any(axis=1)
graph = pd.DataFrame({"entities": list(n_entities.values()) + [n_words],
                      "links": [int((ATTR[:, k] > 0).sum()) for k in range(len(ATTRS))] + [int((TITLE > 0).sum())]},
                     index=list(ATTRS) + ["title word"])
print(graph)
print(f"products {len(wids):,} | knowledge-graph links {graph.links.sum():,}")
print("products with degree > 0: " + " | ".join(f"{s} {(DEG[:, k] > 0).mean():.1%}" for k, s in enumerate(STREAMS)))
print(f"shown rows with attributes {has_attr.mean():.1%} | with title words {(TITLE[ex_p] > 0).any(axis=1).mean():.1%}")
print(f"shown rows cold for CF (step 3) {cold.mean():.1%}, of which with attributes {has_attr[cold].mean():.1%}")
shown_lists = ex.wid.value_counts()                                          # number of lists per product
unseen = ~ex.wid.isin(ex.wid[IN_SPLIT["train"]].unique())                     # product is in no training list
print(f"shown products in only one list: {(shown_lists == 1).mean():.1%}, covering {(ex.wid.map(shown_lists) == 1).mean():.1%} of shown rows")
print(f"rows whose product is in no training list: validation {unseen[IN_SPLIT['valid']].mean():.1%} | test {unseen[IN_SPLIT['test']].mean():.1%}")
del shown_lists, unseen
print(f"query words in the word table: {(QUERY > 0).sum() / (query_ids > 0).sum():.1%} | searches with none: {(QUERY == 0).all(axis=1).mean():.2%}")
ram("section 1")

            entities      links
brand         155707   11588589
shop          232409   11488688
cate1             73   11588589
cate2            634   11588589
cate3           5523   11588589
cate4           6788   11588589
title word    494627  220305753
products 12,319,978 | knowledge-graph links 289,737,386
products with degree > 0: click 13.2% | cart 29.5% | order 16.4%
shown rows with attributes 99.5% | with title words 99.5%
shown rows cold for CF (step 3) 68.2%, of which with attributes 99.4%
shown products in only one list: 75.2%, covering 40.5% of shown rows
rows whose product is in no training list: validation 45.4% | test 45.3%
query words in the word table: 99.4% | searches with none: 0.32%
[section 1] RAM in use: 4.7 GB


## Section 2 · Model inputs

- **History:** per user and stream (click / cart + follow / order), the 50 newest actions, each with its product and age bucket: < 1, 1–3, 3–7, 7–14, 14–30, 30–90, 90–180, 180–365, > 365 days.
- **Lists:** one row per search holding its exposures rows, padded to 200.
- **Step-2 features (28):** Sections 1, 2, 3 and 5. log(1 + x), scaled with the train mean and SD; missing values flagged. Used only by `features_mlp` and `kgin_features`.
- **Same-day rates (8, step-2 Section 6):** left out of the main models. They count clicks from other searches on 17 Oct, some later than the search being scored, and there is no time of day to keep only the earlier ones. Loaded last, for one check in Section 4.

Median actions kept per user: clicks 17, carts 29, orders 15. Lists: 120,209 train, 25,759 validation, 25,760 test.

In [ ]:
HIST_LEN = 50                                  # keeps at most the 50 most recent products each user interacted with for each stream
AGE_EDGES = [1, 3, 7, 14, 30, 90, 180, 365]    # days; 9 age buckets
MAX_LIST = 200                                 # longest shown list
FEATURE_FILES = ["section1_user_product", "section2_affinity", "section3_popularity", "section5_text"]
SAME_DAY_FILE = "section6_label_rates"         # rates from other searches on 17 Oct, some later than the search scored: check only

# 2a. History: H_P[user, stream, slot] = product index (-1 = empty), H_AGE = age bucket. Slot 0 = newest.
hist = hist.sort_values(["sid", "age_s"])
H_P = np.full((N_USERS, len(STREAMS), HIST_LEN), -1, dtype=np.int32)
H_AGE = np.zeros((N_USERS, len(STREAMS), HIST_LEN), dtype=np.int8)
for k, types in enumerate(STREAMS.values()):
    h = hist[hist.type.isin(types)]
    slot = h.groupby("sid").cumcount().to_numpy()
    h, slot = h[slot < HIST_LEN], slot[slot < HIST_LEN]
    H_P[h.sid.to_numpy(), k, slot] = h.p.to_numpy()
    H_AGE[h.sid.to_numpy(), k, slot] = np.searchsorted(AGE_EDGES, h.age_s.to_numpy() / DAY, side="right")
print("median actions kept per user:", {s: int(np.median((H_P[:, k] >= 0).sum(1))) for k, s in enumerate(STREAMS)})
del h, slot; gc.collect()

# 2b. Lists: one row per search holding its exposures row numbers (-1 = padding)
def make_lists(split):
    rows = np.flatnonzero(IN_SPLIT[split])
    searches, list_no = np.unique(SID[rows], return_inverse=True)        # which list each row belongs to
    slot = pd.Series(list_no).groupby(list_no).cumcount().to_numpy()      # position inside its list
    R = np.full((len(searches), MAX_LIST), -1, dtype=np.int64)
    R[list_no, slot] = rows
    return torch.from_numpy(searches.astype(np.int64)).to(DEV), torch.from_numpy(R).to(DEV)

LISTS = {split: make_lists(split) for split in SPLITS}
print("lists:", {split: len(L[0]) for split, L in LISTS.items()})

# 2c. Step-2 features: log(1 + x), scaled with the train mean and SD. NaN stays NaN (the model adds a missing flag).
cols, FEATURE_NAMES = [], []
for f in FEATURE_FILES + [SAME_DAY_FILE]:          # same-day columns go last, so X[:, :N_MAIN] is the main set
    if f == SAME_DAY_FILE:
        N_MAIN = len(FEATURE_NAMES)
    d = pd.read_parquet(f"{FEAT}/{f}.parquet")
    assert (d.sid.to_numpy() == SID).all() and (d.wid.to_numpy() == ex.wid.to_numpy()).all()
    for c in d.columns.drop(["sid", "wid"]):
        x = np.log1p(d[c].to_numpy(dtype=np.float32))
        mu, sd = np.nanmean(x[IN_SPLIT["train"]]), np.nanstd(x[IN_SPLIT["train"]])
        cols.append(((x - mu) / (sd if sd > 0 else 1.0)).astype(np.float16))
        FEATURE_NAMES.append(c)
X = np.stack(cols, axis=1)
del d, x, cols; gc.collect()
print(f"features: {N_MAIN} main + {len(FEATURE_NAMES) - N_MAIN} same-day")
print("main features:", ", ".join(FEATURE_NAMES[:N_MAIN]))

# Stop now if step 2 is out of date: is_cold must follow the before-17-Oct rule, and the features must use the same history as this notebook
is_cold = pd.read_parquet(f"{FEAT}/section3_popularity.parquet", columns=["is_cold"]).is_cold.to_numpy() == 1
assert (is_cold == (DEG[ex_p].sum(axis=1) == 0)).all(), "is_cold breaks the before-17-Oct rule: apply the fix and re-run step 2"
in_history = pd.read_parquet(f"{FEAT}/section1_user_product.parquet", columns=["item_in_history"]).item_in_history.to_numpy() == 1
pair = lambda sid, wid: (np.asarray(sid, np.int64) << 32) | np.asarray(wid, np.int64)
assert (in_history == np.isin(pair(SID, ex.wid), pair(hist.sid, hist.wid))).all(), "step 2's features don't match the history: re-run step 2"
del is_cold, in_history, pair, hist; gc.collect()

# 2d. Everything the model reads goes to the GPU once
gpu = lambda a: torch.from_numpy(a).to(DEV)
ATTR_T, TITLE_T, QUERY_T, DEG_T = gpu(ATTR), gpu(TITLE), gpu(QUERY), gpu(DEG)
H_P_T, H_AGE_T = gpu(H_P), gpu(H_AGE)
EX_P_T, LAB_T, X_T = gpu(ex_p.astype(np.int64)), gpu(LAB.astype(np.int64)), gpu(X)
GAIN_T = torch.tensor(GAIN, dtype=torch.float32, device=DEV)
SHOWN_P = torch.unique(EX_P_T)                 # all shown products (for the random-negatives ablation)
del ATTR, TITLE, QUERY, DEG, H_P, H_AGE, X; gc.collect()
ram("section 2")

median actions kept per user: {'click': 17, 'cart': 29, 'order': 15}
lists: {'train': 120209, 'valid': 25759, 'test': 25760}
features: 28 main + 8 same-day
main features: item_in_history, item_action_7d_count, days_since_item_action_7d, item_cart_90d_count, item_order_1y_count, days_since_item_order_1y, shop_share_7d, brand_share_7d, cate4_share_7d, shop_share_cart_90d, brand_share_cart_90d, cate4_share_cart_90d, shop_share_order_1y, brand_share_order_1y, cate4_share_order_1y, qmatch_cate4_share_7d, qmatch_shop_share_7d, pop_click_7d, pop_click_7d_pct, pop_cart_90d, pop_cart_90d_pct, pop_order_1y, pop_order_1y_pct, is_cold, title_overlap_count, title_overlap_share, title_overlap_rank, title_len
[section 2] RAM in use: 3.3 GB


## Section 3 · Model

- **Product vector:** mean over its links of relation vector × entity vector. Missing links are skipped.
- **Intents (4):** each is a learned mix of the 7 relation vectors. The query sets how much each intent counts.
- **User vector, per stream:** recency-weighted mean of the stream's products, multiplied by the query's intent mix.
- **Score:** a network with one hidden layer of 64 over:

| Input | Size |
|---|---|
| product vector | 64 |
| degree (clicks / carts / orders) | 3 |
| query × product | 64 |
| each stream's user vector × product | 3 × 64 |
| highest cosine with a past product, per stream | 3 |
| step-2 features + missing flags (only when `n_feat > 0`) | 56 (72 with same-day rates) |

**Changes from standard KGIN.**

| Standard KGIN | Here | Reason |
|---|---|---|
| Product ID vectors | Knowledge-graph links + degree | 45.3% of test rows show a product in no training list, so it has no ID vector to learn |
| User ID decides the intents | The query decides | One search per user; test users never appear in training |
| One user vector | One per stream, recency-weighted | Clicks, carts and orders differ in meaning and lifetime |
| Score = user · product | Network over the inputs above | Combines taste, query and popularity |
| BPR: real vs random products | Softmax over the shown list | Learns from products the user saw and skipped |
| Several propagation layers | One pass: attributes → product → user | Links only join products to attributes; keeps 289.7M links practical (deeper passes not tested) |
| Distance-correlation penalty | Squared-cosine penalty | Same purpose, simpler |

**Regularisation:** dropout 0.3 before the network; each product link hidden with probability 0.2 in training; a penalty on similar intents; the best epoch on validation is kept. These replace the usual L2 penalty.

The switches `relations`, `intents`, `query`, `history` and `degree` each remove one part (Section 5). `graph=False` keeps only the features.

In [ ]:
D, N_INTENTS = 64, 4                 # vector size; number of intents
DROPOUT, LINK_DROPOUT = 0.3, 0.2     # dropout before the scoring network; share of a product's links hidden in training
RELATIONS = list(ATTRS) + ["title"]  # the 7 link types

class SearchKGIN(nn.Module):
    """Scores every shown product of a search.
    product vector: mean over the product's links of (relation vector x entity vector)
    intent:         a learned mix of the relation vectors; the query decides how much each intent counts
    user vector:    per stream, recency-weighted mean of history products, seen through the query's intents
    score:          small network over product, degree, query match, stream matches, closest past product
                    (+ step-2 features if n_feat > 0). graph=False keeps only the features."""

    def __init__(self, n_feat=0, graph=True, relations=True, intents=True, query=True, history=True, degree=True):
        super().__init__()
        self.n_feat = n_feat
        self.use = dict(graph=graph, relations=relations, intents=intents, query=query, history=history, degree=degree)
        self.ent = nn.ModuleList([nn.Embedding(n, D, padding_idx=0) for n in n_entities.values()])  # brand, shop, cate1-4
        self.word = nn.Embedding(n_words, D, padding_idx=0)                                         # title and query words
        for table in [*self.ent, self.word]:
            nn.init.normal_(table.weight, std=0.1)
            with torch.no_grad():
                table.weight[0] = 0                                       # row 0 (missing / empty) stays zero
        self.rel = nn.Parameter(1 + 0.1 * torch.randn(len(RELATIONS), D))            # one vector per relation
        self.intent = nn.Parameter(0.1 * torch.randn(N_INTENTS, len(RELATIONS)))     # each intent's weight on each relation
        self.recency = nn.Parameter(torch.zeros(len(STREAMS), len(AGE_EDGES) + 1))   # weight per stream and age bucket

        n_in = 2 * n_feat                                          # features + missing flags
        if graph:
            n_in += D                                              # product vector
            n_in += len(STREAMS) if degree else 0                  # degree
            n_in += D if query else 0                              # query x product
            n_in += len(STREAMS) * (D + 1) if history else 0       # per stream: match vector + closest-product similarity
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(n_in, D), nn.ReLU(), nn.Linear(D, 1))

    def intents(self):
        """alpha [intents, relations] and the intent vectors [intents, D]."""
        alpha = torch.softmax(self.intent, dim=1)
        return alpha, alpha @ self.rel

    def beta(self, q):
        """How much each intent counts for each query [B, intents]."""
        return torch.softmax(q @ self.intents()[1].T, dim=1)

    def product(self, p):
        """Product vectors [..., D] from the knowledge graph. p: product indices, any shape (-1 = padding)."""
        shape = p.shape
        p = p.clamp(min=0).reshape(-1)
        a, t = ATTR_T[p].long(), TITLE_T[p].long()
        title = self.word(t).sum(1) / (t > 0).sum(1, keepdim=True).clamp(min=1)              # mean of title words
        v = torch.stack([emb(a[:, k]) for k, emb in enumerate(self.ent)] + [title], dim=1)   # [N, 7, D] entity vectors
        has = torch.cat([a > 0, (t > 0).any(1, keepdim=True)], dim=1).unsqueeze(2).float()  # [N, 7, 1] link exists
        if self.training:
            has = has * (torch.rand_like(has) >= LINK_DROPOUT)       # hide some links at random
        if self.use["relations"]:
            v = v * self.rel                                         # tag each entity with its relation
        e = (v * has).sum(1) / has.sum(1).clamp(min=1)               # mean over existing links
        return e.reshape(*shape, D)

    def query(self, sid):
        """Mean of the query's word vectors [B, D]."""
        w = QUERY_T[sid].long()
        return self.word(w).sum(1) / (w > 0).sum(1, keepdim=True).clamp(min=1)

    def user(self, sid, q):
        """User vector per stream [B, 3, D], the history product vectors [B, 3, H, D], and which slots are filled."""
        hp = H_P_T[sid].long()
        filled = hp >= 0
        he = self.product(hp)
        age = H_AGE_T[sid].long()
        w = torch.stack([self.recency[k][age[:, k]] for k in range(len(STREAMS))], dim=1)   # weight of each slot's age
        w = torch.softmax(w.masked_fill(~filled, -1e9), dim=2) * filled                      # empty slots -> 0
        h = (w.unsqueeze(3) * he).sum(2)                                                     # recency-weighted mean
        if self.use["intents"]:
            beta = self.beta(q) if self.use["query"] else torch.full((len(sid), N_INTENTS), 1 / N_INTENTS, device=q.device)
            h = h * (beta @ self.intents()[1]).unsqueeze(1)                                   # seen through the active intents
        return h, he, filled

    def forward(self, sid, p, x=None):
        """sid [B] searches, p [B, L] shown products (-1 = padding), x [B, L, n_feat] features. Returns scores [B, L]."""
        parts = [torch.nan_to_num(x.float()), torch.isnan(x).float()] if self.n_feat else []
        if self.use["graph"]:
            e = self.product(p)
            parts.append(e)
            if self.use["degree"]:
                parts.append(DEG_T[p.clamp(min=0)])
            q = self.query(sid)
            if self.use["query"]:
                parts.append(q.unsqueeze(1) * e)
            if self.use["history"]:
                h, he, filled = self.user(sid, q)
                parts.append((h.unsqueeze(1) * e.unsqueeze(2)).flatten(2))                    # match with each stream
                cos = torch.einsum("bld,bshd->blsh", F.normalize(e, dim=-1), F.normalize(he, dim=-1))
                parts.append(cos.masked_fill(~filled.unsqueeze(1), 0).amax(3))                 # closest past product
        return self.head(torch.cat(parts, dim=2)).squeeze(2)

    def independence(self):
        """Penalty that keeps the intents different: squared cosine between every pair."""
        e_p = F.normalize(self.intents()[1], dim=1)
        return ((e_p @ e_p.T).triu(1) ** 2).sum()

# Shape check on 4 training lists
model = SearchKGIN().to(DEV)
sids, R = LISTS["train"]
with torch.no_grad():
    s = model(sids[:4], EX_P_T[R[:4].clamp(min=0)].masked_fill(R[:4] < 0, -1))
print("scores for 4 lists:", tuple(s.shape), "| parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
del model, s

scores for 4 lists: (4, 200) | parameters: 57,350,200


## Section 4 · Training

- **Loss:** per list, cross-entropy between the softmax of the scores and each product's share of the list's gain (order 7, cart 3, click 1), plus 0.01 × the intent penalty.
- 256 lists per step, Adam (learning rate 0.001). Validation after each epoch; the best epoch is kept.
- **Epochs:** 3 for models with the graph, which overfit quickly; 20 for features-only models, which learn more slowly.
- **Models:** `kgin` (no step-2 features), `features_mlp` (28 step-2 features only, same network), `kgin_features` (both).
- **Check only:** `features_mlp_same_day` and `kgin_features_same_day` add the 8 same-day rates. They show what those rates would add if JD could use them in real time; they are not candidates for the final model.

| Model | Train | Valid |
|---|---|---|
| kgin | 0.4019 | 0.3526 |
| features_mlp | 0.4002 | 0.4015 |
| **kgin_features** | 0.4161 | **0.4034** |
| features_mlp_same_day (check) | 0.4139 | 0.4149 |
| kgin_features_same_day (check) | 0.4517 | 0.4131 |
| popularity (step 3) | 0.3326 | 0.3342 |
| NCF (step 3) | 0.3261 | 0.3271 |

- All three main models beat popularity and NCF on validation.
- `features_mlp` alone (0.4015) is well above `kgin` alone (0.3526). Adding KGIN to the features adds 0.0019.
- `kgin` overfits: train is 0.0493 above validation. `features_mlp` has no such gap.
- The KGIN models here peak by epoch 2. After that the loss keeps falling while validation drops, so they are memorising training products.
- `features_mlp` reaches 0.4015 at epoch 5 and stays around 0.401 after that.
- Same-day rates add 0.0134 to `features_mlp`. With them, KGIN no longer helps: `kgin_features_same_day` (0.4131) is below `features_mlp_same_day` (0.4149).

In [ ]:
BATCH, LR, EPOCHS = 256, 1e-3, 3   # lists per step; learning rate; epochs (the best on validation is kept)
FEATURE_EPOCHS = 20                # features-only models learn more slowly, so they get more epochs (the best is still kept)
LAMBDA_IND = 0.01                  # weight of the intent-independence penalty

def get_batch(sid, rows, n_feat):
    """For a batch of lists: product indices, gains (0/1/3/7), the first n_feat features, and which slots are real."""
    valid = rows >= 0
    r = rows.clamp(min=0)
    p = EX_P_T[r].masked_fill(~valid, -1)
    g = GAIN_T[LAB_T[r]] * valid
    return p, g, (X_T[r, :n_feat] if n_feat else None), valid

def list_loss(s, g):
    """Cross-entropy between each list's softmax over scores and each product's share of the list's gain."""
    return -(g / g.sum(1, keepdim=True) * torch.log_softmax(s, dim=1)).sum(1).mean()

@torch.no_grad()
def predict(model, split):
    """Score every shown product of a split (NaN elsewhere), in exposures order."""
    model.eval()
    out = np.full(len(ex), np.nan, dtype=np.float32)
    sids, R = LISTS[split]
    for a in range(0, len(sids), 512):
        p, _, x, valid = get_batch(sids[a:a + 512], R[a:a + 512], model.n_feat)
        s = model(sids[a:a + 512], p, x)
        out[R[a:a + 512][valid].cpu().numpy()] = s[valid].float().cpu().numpy()
    model.train()
    return out

def train(name, n_feat=0, random_neg=False, epochs=EPOCHS, seed=SEED, **use):
    """Train on the shown training lists; score validation after each epoch and keep the best weights."""
    torch.manual_seed(seed)
    model = SearchKGIN(n_feat, **use).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sids, R = LISTS["train"]
    m = IN_SPLIT["valid"]
    best_valid, best_state = -1.0, None
    for epoch in range(epochs):
        t, total = time.time(), 0.0
        order = torch.randperm(len(sids), device=DEV)
        for a in range(0, len(sids), BATCH):
            b = order[a:a + BATCH]
            p, g, x, valid = get_batch(sids[b], R[b], n_feat)
            if random_neg:                                  # ablation: ignored products replaced by random products
                swap = valid & (g == 0)
                p[swap] = SHOWN_P[torch.randint(len(SHOWN_P), (int(swap.sum()),), device=DEV)]
            s = model(sids[b], p, x).masked_fill(~valid, -1e9)   # padding gets no probability
            loss = list_loss(s, g) + LAMBDA_IND * model.independence()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(b)
        valid = ndcg_at_k(SID[m], LAB[m], predict(model, "valid")[m])
        print(f"[{name}] epoch {epoch + 1}/{epochs}: loss {total.item() / len(sids):.4f} | valid NDCG@10 {valid:.4f} | {time.time() - t:.0f}s")
        if valid > best_valid:
            best_valid, best_state = valid, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model

def fit(name, save=True, **kw):
    """Train, then score train / valid / test with the kept weights. save=True writes the weights and scores."""
    t = time.time()
    model = train(name, **kw)
    score = np.full(len(ex), np.nan, dtype=np.float32)
    for split in SPLITS:
        score[IN_SPLIT[split]] = predict(model, split)[IN_SPLIT[split]]
    evaluate(name, score)
    if save:
        torch.save(model.state_dict(), f"{MODEL}/{name}.pt")
        save_scores(name, score)
    print(f"{name} done in {(time.time() - t) / 60:.1f} min")
    return model

kgin = fit("kgin")                                                     # graphs, query, history; no step-2 features
fit("features_mlp", n_feat=N_MAIN, graph=False, epochs=FEATURE_EPOCHS)  # step-2 features only, same network
kgin_features = fit("kgin_features", n_feat=N_MAIN)                    # KGIN + step-2 features
# Check only: what the same-day rates would add if JD could use them
fit("features_mlp_same_day", n_feat=len(FEATURE_NAMES), graph=False, epochs=FEATURE_EPOCHS)
fit("kgin_features_same_day", n_feat=len(FEATURE_NAMES))
ram("section 4")
print(pd.DataFrame(results).set_index("model").round(4))

[kgin] epoch 1/3: loss 3.8668 | valid NDCG@10 0.3516 | 45s
[kgin] epoch 2/3: loss 3.6819 | valid NDCG@10 0.3526 | 46s
[kgin] epoch 3/3: loss 3.5528 | valid NDCG@10 0.3506 | 48s
                  kgin   train 0.4019 | valid 0.3526
kgin done in 3.4 min
[features_mlp] epoch 1/20: loss 3.6633 | valid NDCG@10 0.3980 | 5s
[features_mlp] epoch 2/20: loss 3.5737 | valid NDCG@10 0.3989 | 8s
[features_mlp] epoch 3/20: loss 3.5592 | valid NDCG@10 0.4000 | 5s
[features_mlp] epoch 4/20: loss 3.5521 | valid NDCG@10 0.4000 | 6s
[features_mlp] epoch 5/20: loss 3.5441 | valid NDCG@10 0.4015 | 5s
[features_mlp] epoch 6/20: loss 3.5406 | valid NDCG@10 0.4011 | 4s
[features_mlp] epoch 7/20: loss 3.5379 | valid NDCG@10 0.4011 | 5s
[features_mlp] epoch 8/20: loss 3.5361 | valid NDCG@10 0.4006 | 5s
[features_mlp] epoch 9/20: loss 3.5354 | valid NDCG@10 0.4009 | 5s
[features_mlp] epoch 10/20: loss 3.5346 | valid NDCG@10 0.4013 | 5s
[features_mlp] epoch 11/20: loss 3.5216 | valid NDCG@10 0.4009 | 5s
[features_

## Section 5 · Ablations

Each variant removes one part of `kgin` and is trained with two seeds. The gap between seeds shows chance variation.

| Variant | Seed 4222 | Seed 4223 | Mean | Change | Seed gap |
|---|---|---|---|---|---|
| kgin | 0.3526 | 0.3559 | 0.3542 | — | 0.0033 |
| no relation vectors | 0.3512 | 0.3530 | 0.3521 | −0.0021 | 0.0018 |
| no intents | 0.3526 | 0.3561 | 0.3544 | +0.0001 | 0.0035 |
| no query | 0.3547 | 0.3566 | 0.3557 | +0.0014 | 0.0018 |
| **no history** | 0.3475 | 0.3465 | 0.3470 | **−0.0072** | 0.0010 |
| **no degree** | 0.2774 | 0.2782 | 0.2778 | **−0.0764** | 0.0008 |
| **random negatives** | 0.3419 | 0.3403 | 0.3411 | **−0.0131** | 0.0016 |

- Seeds differ by 0.0020 on average and at most 0.0035, so only larger changes count.
- Degree matters most. Without it, KGIN falls far below popularity (0.3342). This model was still improving at epoch 3 (0.2484 → 0.2644 → 0.2774), so more epochs may shrink the drop a little.
- Training on the shown lists matters: with random products instead, the loss drops fast (2.63 → 1.02) while validation falls (0.3419 → 0.3157).
- History adds 0.0072.
- Relation vectors (−0.0021) are too close to the seed gap to call. Intents make no difference (+0.0001). Removing the query doesn't hurt (+0.0014).

**KGIN's gain over the features.** The same comparison with a second seed for both models.

| | Seed 4222 | Seed 4223 |
|---|---|---|
| kgin_features | 0.4034 | 0.4042 |
| features_mlp | 0.4015 | 0.4018 |
| **KGIN's gain** | **0.0019** | **0.0024** |

The gain holds on both seeds and is larger than either model's seed gap (0.0008 and 0.0003). But it is small.

In [ ]:
SEEDS = [SEED, SEED + 1]                           # every variant is trained twice; the gap between seeds shows chance variation
VARIANTS = {
    "kgin": {},                                    # full model (seed SEED already trained in Section 4)
    "no_relation_vectors": dict(relations=False),  # product = plain mean of its entities
    "no_intents": dict(intents=False),             # history not weighted by intents
    "no_query": dict(query=False),
    "no_history": dict(history=False),
    "no_degree": dict(degree=False),               # no interaction graph
    "random_negatives": dict(random_neg=True),     # ignored products replaced by random products
}
for name, kw in VARIANTS.items():
    for seed in SEEDS:
        if not (name == "kgin" and seed == SEED):
            fit(f"{name} (seed {seed})", save=False, seed=seed, **kw)

valid = pd.DataFrame(results).set_index("model").valid
valid["kgin (seed %d)" % SEED] = valid["kgin"]
ablation = pd.DataFrame({f"seed {s}": [valid[f"{n} (seed {s})"] for n in VARIANTS] for s in SEEDS}, index=list(VARIANTS))
ablation["mean"] = ablation.mean(axis=1)
ablation["change vs kgin"] = ablation["mean"] - ablation.loc["kgin", "mean"]
ablation["seed gap"] = (ablation[f"seed {SEEDS[0]}"] - ablation[f"seed {SEEDS[1]}"]).abs()
ablation.round(4).to_csv(f"{MODEL}/ablations.csv")
print(ablation.round(4))
print(f"average seed gap: {ablation['seed gap'].mean():.4f}")

# Is KGIN's gain over the features real? Second seed for both models; the first seed is from Section 4.
fit(f"kgin_features (seed {SEEDS[1]})", save=False, seed=SEEDS[1], n_feat=N_MAIN)
fit(f"features_mlp (seed {SEEDS[1]})", save=False, seed=SEEDS[1], n_feat=N_MAIN, graph=False, epochs=FEATURE_EPOCHS)
valid = pd.DataFrame(results).set_index("model").valid
gain = pd.DataFrame({f"seed {SEEDS[0]}": [valid[n] for n in ["kgin_features", "features_mlp"]],
                     f"seed {SEEDS[1]}": [valid[f"{n} (seed {SEEDS[1]})"] for n in ["kgin_features", "features_mlp"]]},
                    index=["kgin_features", "features_mlp"])
gain.loc["KGIN's gain"] = gain.loc["kgin_features"] - gain.loc["features_mlp"]
gain.round(4).to_csv(f"{MODEL}/kgin_gain_check.csv")
print(gain.round(4))
ram("section 5")

[kgin (seed 4223)] epoch 1/3: loss 3.8477 | valid NDCG@10 0.3532 | 47s
[kgin (seed 4223)] epoch 2/3: loss 3.6725 | valid NDCG@10 0.3557 | 47s
[kgin (seed 4223)] epoch 3/3: loss 3.5349 | valid NDCG@10 0.3559 | 48s
      kgin (seed 4223)   train 0.4887 | valid 0.3559
kgin (seed 4223) done in 3.1 min
[no_relation_vectors (seed 4222)] epoch 1/3: loss 3.8611 | valid NDCG@10 0.3512 | 45s
[no_relation_vectors (seed 4222)] epoch 2/3: loss 3.6552 | valid NDCG@10 0.3508 | 44s
[no_relation_vectors (seed 4222)] epoch 3/3: loss 3.5467 | valid NDCG@10 0.3475 | 44s
no_relation_vectors (seed 4222)   train 0.3661 | valid 0.3512
no_relation_vectors (seed 4222) done in 2.9 min
[no_relation_vectors (seed 4223)] epoch 1/3: loss 3.8420 | valid NDCG@10 0.3527 | 44s
[no_relation_vectors (seed 4223)] epoch 2/3: loss 3.6470 | valid NDCG@10 0.3530 | 44s
[no_relation_vectors (seed 4223)] epoch 3/3: loss 3.5403 | valid NDCG@10 0.3530 | 44s
no_relation_vectors (seed 4223)   train 0.4430 | valid 0.3530
no_relation_v

## Section 6 · Test results

Each model's saved scores are scored once on test (25,760 searches). Paired tests compare two models on the same searches.

| Model | Train | Valid | Test |
|---|---|---|---|
| random | 0.0809 | 0.0815 | 0.0804 |
| popularity | 0.3326 | 0.3342 | 0.3329 |
| MF | 0.3158 | 0.3159 | 0.3168 |
| NCF | 0.3261 | 0.3271 | 0.3290 |
| NMF | 0.3256 | 0.3260 | 0.3267 |
| kgin | 0.4019 | 0.3526 | 0.3541 |
| features_mlp | 0.4002 | 0.4015 | 0.4021 |
| **kgin_features** | 0.4161 | 0.4034 | **0.4060** |
| features_mlp_same_day (check) | 0.4139 | 0.4149 | 0.4158 |
| kgin_features_same_day (check) | 0.4517 | 0.4131 | 0.4150 |

| Comparison | Difference | 95% interval | p-value | Better / worse searches |
|---|---|---|---|---|
| kgin − popularity | +0.0212 | +0.0189 to +0.0235 | < 0.0001 | 29% / 22% |
| kgin − NCF | +0.0251 | +0.0226 to +0.0276 | < 0.0001 | 31% / 23% |
| kgin_features − features_mlp | +0.0039 | +0.0024 to +0.0053 | < 0.0001 | 20% / 18% |
| kgin_features − popularity | +0.0731 | +0.0700 to +0.0762 | < 0.0001 | 36% / 21% |
| features_mlp_same_day − features_mlp | +0.0136 | +0.0117 to +0.0156 | < 0.0001 | 25% / 22% |

- `kgin_features` is the best main model: test 0.4060, which is 0.0731 above popularity.
- KGIN alone beats every baseline (0.3541 vs popularity 0.3329), but `features_mlp` alone does better (0.4021).
- KGIN adds 0.0039 on top of the features. The interval stays above 0, so this is unlikely to be chance, but it is small.
- The same-day rates add 0.0136 to `features_mlp`, more than KGIN does. They stay out of the final model because some come from after the search being scored. With them, KGIN no longer helps: 0.4150 vs 0.4158.

In [ ]:
BASELINES = ["random", "popularity", "mf", "ncf", "nmf"]   # step 3
FINAL = ["kgin", "features_mlp", "kgin_features", "features_mlp_same_day", "kgin_features_same_day"]
m = IN_SPLIT["test"]

# Test NDCG@10 of every search, from the saved score files
per_search = {name: ndcg_by_search(SID[m], LAB[m], load_scores(BASE, name)[m]) for name in BASELINES}
per_search.update({name: ndcg_by_search(SID[m], LAB[m], load_scores(MODEL, name)[m]) for name in FINAL})

table = pd.concat([pd.read_csv(f"{BASE}/results.csv", index_col=0)[["train", "valid"]],
                   pd.DataFrame(results).set_index("model").loc[FINAL, ["train", "valid"]]])
table["test"] = pd.Series({name: s.mean() for name, s in per_search.items()})
table.to_csv(f"{MODEL}/results.csv")
print(f"test searches: {len(per_search['kgin']):,}")
display(table.round(4))

# Paired comparisons on the same test searches
rows = []
for a, b in [("kgin", "popularity"), ("kgin", "ncf"), ("kgin_features", "features_mlp"), ("kgin_features", "popularity"),
             ("features_mlp_same_day", "features_mlp")]:
    diff = per_search[a] - per_search[b]
    half = 1.96 * diff.std() / np.sqrt(len(diff))                       # 95% interval half-width
    rows.append({"comparison": f"{a} - {b}", "difference": diff.mean(), "95% low": diff.mean() - half,
                 "95% high": diff.mean() + half, "p-value": stats.ttest_rel(per_search[a], per_search[b]).pvalue,
                 "share better": (diff > 0).mean(), "share worse": (diff < 0).mean()})
paired = pd.DataFrame(rows).set_index("comparison")
paired.to_csv(f"{MODEL}/paired_tests.csv")
display(paired.round(4))

test searches: 25,760


,train,valid,test
random,0.0809,0.0815,0.0804
popularity,0.3326,0.3342,0.3329
mf,0.3158,0.3159,0.3168
ncf,0.3261,0.3271,0.3290
nmf,0.3256,0.3260,0.3267
kgin,0.4019,0.3526,0.3541
features_mlp,0.4002,0.4015,0.4021
kgin_features,0.4161,0.4034,0.4060
features_mlp_same_day,0.4139,0.4149,0.4158
kgin_features_same_day,0.4517,0.4131,0.4150


,difference,95% low,95% high,p-value,share better,share worse
comparison,,,,,,
kgin - popularity,0.0212,0.0189,0.0235,0.0,0.2875,0.2218
kgin - ncf,0.0251,0.0226,0.0276,0.0,0.3130,0.2306
kgin_features - features_mlp,0.0039,0.0024,0.0053,0.0,0.2049,0.1781
kgin_features - popularity,0.0731,0.0700,0.0762,0.0,0.3614,0.2054
features_mlp_same_day - features_mlp,0.0136,0.0117,0.0156,0.0,0.2539,0.2183


## Section 7 · Interpretation

Validation only. The printed tables below are the results.

### 7.1 Intents
Each row shows how much one intent weights each of the 7 relations.

Each intent specialises: 0 = brand (with categories 2–3), 1 = shop, 2 = title words (with shop), 3 = categories 2–4.

### 7.2 Intent per query
- Strongest intent: intent 3 for 11,733 searches, intent 1 for 6,993, intent 2 for 3,753, intent 0 for 3,280.
- The strongest intent gets only 0.27–0.29 on average, close to an even split (0.25). Queries barely change the intent mix, which fits Section 5: removing intents changes nothing.

### 7.3 Recency
Weight of each age bucket, relative to the last day. History stops at each stream's window (clicks 7 days, carts 90, orders 365), so weights past the window barely train: ignore them.

- Clicks fade fast: one from 1–3 days ago counts 0.23 of one from the last day, and 3–7 days 0.18.
- Carts drop to 0.47–0.70 and stay there.
- Orders don't fade: 1.03–1.26 at every age.

### 7.4 Step 3's groups
Validation NDCG@10 per group, for popularity, NCF, KGIN and KGIN + features.

- Where CF could score none of the interacted products, KGIN gets 0.1316 (NCF 0.0737, popularity 0.1012). With the features, 0.2041.
- KGIN beats popularity in every activity group, by 0.012 to 0.023. The features add least for users with no clicks in the last 7 days (+0.024, against +0.055 to +0.060 for the rest).
- Repeat searches gain most from the features: +0.124, against +0.040 for new searches. KGIN alone does worse on repeat searches than new ones (0.3322 vs 0.3555); with the features it does better (0.4565 vs 0.3957).

### Summary

| Step-3 weakness | Change | Result |
|---|---|---|
| Cold products | Product vectors from the knowledge graph | Helped. "None" group: 0.1316 vs NCF 0.0737 |
| Learns popularity, not taste | Train on the shown lists | Helped. Random negatives: −0.0131. But degree is still the biggest input (no degree: −0.0764) |
| History not used well | Per-stream, recency-weighted history | Helped. No history: −0.0072 |
| Ignores the query | Query vector and query-weighted intents | No gain. No query: +0.0014; no intents: +0.0001 |

KGIN beats every step-3 baseline without the step-2 features (test 0.3541 vs popularity 0.3329), but the features alone do better (0.4021). Together they score highest (0.4060). KGIN adds only 0.0039 on top of the features, so most of the strength comes from the features.

In [ ]:
m = IN_SPLIT["valid"]                       # validation only; test stays untouched
POS = LAB > LABEL["none"]                   # interacted rows
kgin.eval()

# 7.1 Intents: weight of each relation in each intent
alpha = pd.DataFrame(kgin.intents()[0].detach().cpu().numpy(), columns=RELATIONS,
                     index=[f"intent {p}" for p in range(N_INTENTS)])
print("7.1 Intent weights (rows sum to 1)"); display(alpha.round(2))

# 7.2 The intent each validation query weights most
sids, _ = LISTS["valid"]
with torch.no_grad():
    beta = torch.cat([kgin.beta(kgin.query(sids[a:a + 4096])) for a in range(0, len(sids), 4096)]).cpu().numpy()
top = pd.Series([f"intent {p}" for p in beta.argmax(1)])
print("7.2 Searches by their strongest intent")
display(pd.DataFrame({"searches": top.value_counts(), "mean weight": pd.Series(beta.max(1)).groupby(top).mean()}).sort_index().round(2))

# 7.3 Recency: weight of each age bucket relative to the last day
rec = torch.exp(kgin.recency - kgin.recency[:, :1]).detach().cpu().numpy()
ages = [f"<{AGE_EDGES[0]}d"] + [f"{a}-{b}d" for a, b in zip(AGE_EDGES[:-1], AGE_EDGES[1:])] + [f">{AGE_EDGES[-1]}d"]
print("7.3 Recency weight relative to the last day"); display(pd.DataFrame(rec, index=list(STREAMS), columns=ages).round(2))

# 7.4 Step 3's diagnostic groups
scores = {"popularity": load_scores(BASE, "popularity"), "ncf": load_scores(BASE, "ncf"),
          "kgin": load_scores(MODEL, "kgin"), "kgin_features": load_scores(MODEL, "kgin_features")}

def by_group(group, order):
    """Validation NDCG@10 per group. `group` maps each search id to a group name."""
    g_row = group.reindex(SID).to_numpy()
    out = []
    for g in order:
        s = m & (g_row == g)
        out.append({"group": g, "searches": len(np.unique(SID[s])), **{n: ndcg_at_k(SID[s], LAB[s], sc[s]) for n, sc in scores.items()}})
    return pd.DataFrame(out).set_index("group").round(4)

share = pd.Series(~cold[m & POS]).groupby(SID[m & POS]).mean()          # share of interacted products CF could score
cf_cover = pd.Series(np.select([share == 1, share == 0], ["all", "none"], "some"), index=share.index)
ctx = pd.read_parquet(f"{FEAT}/section4_context.parquet").set_index("sid")
activity = pd.cut(ctx.clicks_7d, [-1, 0, 20, 100, np.inf], labels=["0", "1-20", "21-100", "over 100"]).astype(str)
qm = pd.read_parquet(f"{FEAT}/section2_affinity.parquet", columns=["sid", "qmatch_cate4_share_7d"])
repeat = qm.qmatch_cate4_share_7d.notna().groupby(qm.sid).any().map({True: "repeat", False: "new"})
groups = {"interacted products CF could score": (cf_cover, ["all", "some", "none"]),
          "clicks in the last 7 days": (activity, ["0", "1-20", "21-100", "over 100"]),
          "search": (repeat, ["new", "repeat"])}
seg = {}
for title, (group, order) in groups.items():
    seg[title] = by_group(group, order)
    print(f"7.4 {title}"); display(seg[title])

pd.concat(seg, names=["grouping"]).to_csv(f"{MODEL}/segments.csv")
alpha.round(4).to_csv(f"{MODEL}/intent_weights.csv")
del scores, qm, ctx; gc.collect(); ram("section 7")

7.1 Intent weights (rows sum to 1)


,brand,shop,cate1,cate2,cate3,cate4,title
intent 0,0.47,0.02,0.04,0.15,0.27,0.02,0.02
intent 1,0.03,0.81,0.03,0.03,0.03,0.03,0.04
intent 2,0.03,0.25,0.09,0.02,0.02,0.02,0.55
intent 3,0.02,0.02,0.02,0.19,0.41,0.31,0.02


7.2 Searches by their strongest intent


,searches,mean weight
intent 0,3280,0.27
intent 1,6993,0.28
intent 2,3753,0.28
intent 3,11733,0.29


7.3 Recency weight relative to the last day


,<1d,1-3d,3-7d,7-14d,14-30d,30-90d,90-180d,180-365d,>365d
click,1.0,0.23,0.18,0.38,0.38,0.38,0.38,0.38,0.38
cart,1.0,0.70,0.59,0.56,0.61,0.47,0.59,0.62,0.62
order,1.0,1.03,1.14,1.17,1.20,1.03,1.03,1.26,1.11


... (3 outputs left out)

,searches,popularity,ncf,kgin,kgin_features
group,,,,,
0,4184,0.3468,0.3347,0.3585,0.3827
1-20,9648,0.3336,0.3291,0.3564,0.4112
21-100,8605,0.3321,0.3253,0.3501,0.4060
over 100,3322,0.3206,0.3126,0.3398,0.3997


7.4 search


,searches,popularity,ncf,kgin,kgin_features
group,,,,,
new,22496,0.3386,0.3313,0.3555,0.3957
repeat,3263,0.3032,0.2978,0.3322,0.4565


[section 7] RAM in use: 3.4 GB


## Section 8 · The past project's kind of list

The past project (Group 3, LSTM + hybrid NCF) reported NDCG@10 = 0.5684, but on short lists: every interacted product plus up to 4 random ignored ones for each, so a median of 5 products. Our lists are full (median 90), so the two numbers can't be compared directly.

Here step 3's rule (Section 1) rebuilds short lists from our test searches, and the saved scores of each model are ranked on them. No model is retrained. The LightGBM notebook (Section 9) builds the same lists, so random should print 0.5341 on validation and 0.5333 on test, as it does there. Their exact lists and test users differ from ours, so this compares the same kind of list, not the same lists.

In [ ]:
PAST_REPORTED = 0.5684   # past project's best model (LSTM + hybrid NCF), printed in 3.1_Hybrid_NCF_with_LSTM.ipynb cell 94
PAST_MODELS = ["kgin", "features_mlp", "kgin_features"]
past_scores = {name: load_scores(MODEL, name) for name in PAST_MODELS}

def short_lists(split):
    """Step 3's rule: every interacted product plus 4 random ignored ones per interacted product."""
    m = IN_SPLIT[split]
    d = ex.loc[m, ["sid", "label"]].assign(**{name: s[m] for name, s in past_scores.items()})
    pos, neg = d[d.label > 0], d[d.label == 0]
    neg = neg.assign(r=np.random.default_rng(SEED).random(len(neg))).sort_values(["sid", "r"])
    n_pos = pos.groupby("sid").size()
    keep = neg.groupby("sid").cumcount().to_numpy() < 4 * n_pos.reindex(neg.sid).to_numpy()
    return pd.concat([pos, neg[keep].drop(columns="r")]).sort_values("sid", kind="stable")

def random_ndcg(d, draws=5):
    """Average NDCG@10 of a random order over a few draws (as in step 3)."""
    return np.mean([ndcg_at_k(d.sid.to_numpy(), d.label.to_numpy(), np.random.default_rng(i).random(len(d))) for i in range(draws)])

short_va = short_lists("valid")
print(f"validation short lists: random {random_ndcg(short_va):.4f} (step 3: 0.5341)")

short = short_lists("test")
past = pd.Series({"random": random_ndcg(short), "past project (as reported)": PAST_REPORTED,
                  **{name: ndcg_at_k(short.sid.to_numpy(), short.label.to_numpy(), short[name].to_numpy()) for name in PAST_MODELS}},
                 name="test NDCG@10, short lists")
past.to_csv(f"{MODEL}/past_project_lists.csv")
print(f"test short lists: {short.sid.nunique():,} searches | median size {short.groupby('sid').size().median():.0f}")
print(past.round(4).to_string())
print(f"kgin_features - past project: {past['kgin_features'] - PAST_REPORTED:+.4f}")

validation short lists: random 0.5341 (step 3: 0.5341)
test short lists: 25,760 searches | median size 5
random                        0.5333
past project (as reported)    0.5684
kgin                          0.7591
features_mlp                  0.7776
kgin_features                 0.7825
kgin_features - past project: +0.2141
